<a href="https://colab.research.google.com/github/chamorrosandra214-rgb/BOTELLAS-DESCARTADAS/blob/main/EJERCICIO%204%20Botellas%20Descartadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install SimPy if not already installed
!pip install simpy

import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1
PROB_DEFECTO = 0.15        # probabilidad de que una botella sea defectuosa

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)

        # Decisión: ¿aprobada o defectuosa?
        if random.random() < PROB_DEFECTO:
            descartadas.append(id_botella)
            print(f"❌ Botella {id_botella} descartada en {env.now:.1f}")
            return
        else:
            print(f"✅ Botella {id_botella} aprobada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"📦 Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque, descartadas):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []
descartadas = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque, descartadas))
env.run(until=500)  # simular 500 segundos

# Resumen final
print("\n--- RESULTADOS ---")
print(f"Botellas descartadas: {len(descartadas)}")
print(f"IDs descartados: {descartadas}")

Botella 1 moldeada en 31.2
Botella 2 moldeada en 44.2
Botella 3 moldeada en 84.5
Botella 1 enfriada en 91.2
Botella 4 moldeada en 99.1
Botella 2 enfriada en 104.2
Botella 5 moldeada en 113.1
Botella 6 moldeada en 133.0
Botella 3 enfriada en 144.5
Botella 7 moldeada en 149.8
Botella 4 enfriada en 159.1
Botella 5 enfriada en 173.1
Botella 9 moldeada en 181.9
Botella 8 moldeada en 190.3
Botella 6 enfriada en 193.0
✅ Botella 1 aprobada en 208.8
Botella 7 enfriada en 209.8
Botella 10 moldeada en 213.5
✅ Botella 2 aprobada en 232.0
Botella 11 moldeada en 236.7
Botella 12 moldeada en 241.2
Botella 9 enfriada en 241.9
✅ Botella 3 aprobada en 244.6
Botella 8 enfriada en 250.3
Botella 13 moldeada en 266.9
Botella 10 enfriada en 273.5
Botella 14 moldeada en 275.7
Botella 11 enfriada en 296.7
Botella 12 enfriada en 301.2
Botella 15 moldeada en 320.0
Botella 13 enfriada en 326.9
✅ Botella 4 aprobada en 335.3
Botella 14 enfriada en 335.7
✅ Botella 5 aprobada en 364.4
Botella 15 enfriada en 380.0
Bot